[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/06_linear_regression/92_assignment_linear_regression_multiple_variables.ipynb)

# Assignment 92 — Linear Regression with Multiple Variables

Bài toán: dự đoán **giá nhà** (Portland, Oregon) từ **diện tích** (feet²) và **số phòng ngủ** — bài tập kinh điển của khoá Andrew Ng. Trước khi làm, hãy học bài [02](02_linear_regression_multiple_variables.ipynb) và [03](03_normal_equation_and_gd_tips.ipynb).

### Hướng dẫn làm bài
- Dùng NumPy, vector hoá, không vòng lặp trên dữ liệu.
- Viết code trong `### START CODE HERE ###` … `### END CODE HERE ###`, chạy cell kiểm tra.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/06_linear_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.checks import check_close, check_less, check_shape, check_true
from ml_course.utils import repo_path

data = np.loadtxt(repo_path("data", "sample", "ex3_house_multivariate.txt"), delimiter=",")
X_raw, y = data[:, :2], data[:, 2]
m = len(y)
print("X (5 dòng đầu):\n", X_raw[:5])
print("y (5 giá trị đầu):", y[:5])

## 1. Trực quan hoá dữ liệu

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_raw[:, 0], y, s=15)
axes[0].set_xlabel("Size (feet²)")
axes[0].set_ylabel("Price ($)")
axes[1].scatter(X_raw[:, 1], y, s=15)
axes[1].set_xlabel("Bedrooms")
plt.show()
print("Thang đo: diện tích ~", X_raw[:, 0].mean().round(), "| số phòng ~", X_raw[:, 1].mean().round(1))

## 2. Chuẩn hoá dữ liệu (feature normalization)

Diện tích lớn gấp ~1 000 lần số phòng → chuẩn hoá z-score để GD hội tụ nhanh:

$$x_j := \frac{x_j - \mu_j}{\sigma_j}$$

Viết `feature_normalize(X)` trả về `(X_norm, mu, sigma)` — lưu `mu`, `sigma` để chuẩn hoá dữ liệu mới khi dự đoán.

In [ ]:
def feature_normalize(X):
    ### START CODE HERE ### (≈ 3 dòng)
    mu = None
    sigma = None
    X_norm = None
    ### END CODE HERE ###
    return X_norm, mu, sigma


X_norm, mu, sigma = feature_normalize(X_raw)
print("mu =", mu, "| sigma =", sigma)

In [ ]:
check_close(X_norm.mean(axis=0), [0, 0], tol=1e-9, message="mean = 0")
check_close(X_norm.std(axis=0), [1, 1], tol=1e-9, message="std = 1")

Thêm cột $x_0 = 1$:

In [ ]:
### START CODE HERE ### (≈ 1 dòng)
X = None
### END CODE HERE ###

In [ ]:
check_shape(X, (47, 3))

## 3. Cost function và Gradient Descent (vector hoá)

$$J(\theta) = \frac{1}{2m}(X\theta - y)^\top(X\theta - y), \qquad \theta := \theta - \frac{\alpha}{m}X^\top(X\theta - y)$$

In [ ]:
def compute_cost(X, y, theta):
    ### START CODE HERE ### (≈ 2 dòng)
    err = None
    J = None
    ### END CODE HERE ###
    return J


def gradient_descent(X, y, theta, alpha, num_iters):
    J_history = []
    for _ in range(num_iters):
        ### START CODE HERE ### (≈ 1 dòng)
        theta = None
        ### END CODE HERE ###
        J_history.append(compute_cost(X, y, theta))
    return theta, J_history


theta, J_history = gradient_descent(X, y, np.zeros(3), alpha=0.1, num_iters=400)
print("θ =", theta)

In [ ]:
check_close(compute_cost(X, y, np.zeros(3)), 65591548106.45744, tol=1e-6, message="J(0)")
theta_ne = np.linalg.lstsq(X, y, rcond=None)[0]
check_close(theta, theta_ne, tol=1e-3, message="GD hội tụ về nghiệm tối ưu")

## 4. Nghiệm với scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression

sk = LinearRegression().fit(X_norm, y)
print("sklearn: intercept =", sk.intercept_, "| coef =", sk.coef_)

## 5. Tuỳ chọn learning rate

Chạy GD 50 vòng với các learning rate `[0.3, 0.1, 0.03, 0.01]`, vẽ $J$ theo vòng lặp trên cùng một hình. Learning rate nào hội tụ nhanh nhất mà vẫn ổn định?

In [ ]:
fig, ax = plt.subplots()
histories = {}
for alpha in [0.3, 0.1, 0.03, 0.01]:
    ### START CODE HERE ### (≈ 2 dòng)
    hist = None
    ### END CODE HERE ###
    ax.plot(histories[alpha], label=f"α = {alpha}")
ax.set_xlabel("vòng lặp")
ax.set_ylabel("J")
ax.legend()
plt.show()

In [ ]:
check_true(histories[0.3][-1] < histories[0.01][-1], "α = 0.3 hội tụ nhanh hơn α = 0.01 trong 50 vòng")

## 6. Dự đoán

Dự đoán giá căn nhà **1650 feet², 3 phòng ngủ**. ⚠️ Phải **chuẩn hoá** input bằng `mu`, `sigma` của dữ liệu train trước khi nhân với θ!

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
x_new = None
price = None
### END CODE HERE ###
print(f"Giá dự đoán cho nhà 1650 feet², 3 phòng ngủ: ${price:,.0f}")

In [ ]:
check_close(price, 293081.46, tol=1e-3, message="giá dự đoán ≈ $293,081")

**Đầu ra kỳ vọng:** khoảng `$293,081`

## 7. Normal Equation

Không cần chuẩn hoá, không cần learning rate, không vòng lặp:

$$\theta = (X^\top X)^{-1}X^\top y$$

Dùng **dữ liệu gốc** (chưa chuẩn hoá) + cột 1, giải bằng `np.linalg.solve` (không dùng `inv`). Dự đoán lại cho nhà 1650 feet², 3 phòng ngủ — kết quả phải **giống** cách dùng gradient descent.

In [ ]:
X_raw_b = np.hstack([np.ones((m, 1)), X_raw])
### START CODE HERE ### (≈ 2 dòng)
theta_normal = None
price_normal = None
### END CODE HERE ###
print("θ (normal equation, dữ liệu gốc) =", theta_normal)
print(f"Giá dự đoán: ${price_normal:,.0f}")

In [ ]:
check_close(price_normal, price, tol=1e-4, message="normal equation và GD cho cùng dự đoán")

### Tổng kết

- Hồi quy tuyến tính đa biến vector hoá: $J(\theta)$ và gradient descent một dòng.
- Chuẩn hoá feature (lưu $\mu, \sigma$ để dùng khi dự đoán), chọn learning rate bằng cách quét và vẽ $J$.
- Normal equation cho cùng kết quả, không cần chuẩn hoá — hệ số khác nhau vì thang đo khác nhau, nhưng **dự đoán** giống hệt.

---
<sub>Bài gốc: *Programming Assignment: Linear Regression with Multiple Variables* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 3 (MIT License), dựa trên bài tập của Andrew Ng. Đã thêm kiểm tra tự động, dùng `solve` thay vì nghịch đảo.</sub>